## 1. Imports and Paths

In [1]:
from pathlib import Path
import json
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Ajusta estas rutas a tu entorno actual

print("Current working directory:", os.getcwd())

#PROJECT_ROOT = Path("/home/victor/gw/cbc_pe")
#DATA_ROOT = Path("/home/victor/gw/cbc_pe/data")
PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe")
DATA_ROOT = Path("/data/vserrano/cbc_pe_data")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"PROJECT_ROOT does not exist: {PROJECT_ROOT}")
else:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_ID = "bbh_processed_4s_seobnrv4opt_snr10-25_n500_000"

CONFIG_DIR = PROJECT_ROOT / "configs" 
MODEL_DIR = DATA_ROOT / "models" / "checkpoints" / DATASET_ID
RESULTS_DIR = DATA_ROOT / "results" / DATASET_ID

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CONFIG_DIR exists:", CONFIG_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("RESULTS_DIR exists:", RESULTS_DIR.exists())

Current working directory: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks
PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
DATA_ROOT: /data/vserrano/cbc_pe_data
CONFIG_DIR exists: True
MODEL_DIR exists: True
RESULTS_DIR exists: True


In [2]:
generation_config_path =  PROJECT_ROOT / "configs" / "generation" / "generate_500k_bbh_4s.json"
training_config_path = PROJECT_ROOT / "configs" / "experiments" / "train_500k_M08_resdilated_emb64_d124_bs256_seed123.json"

with open(generation_config_path, "r") as f:
    gen_cfg = json.load(f)

with open(training_config_path, "r") as f:
    train_cfg = json.load(f)

print("Generation config keys:", gen_cfg.keys())
print("Training config keys:", train_cfg.keys())
print("Training parameters:", train_cfg["model"]["kwargs"].keys())

Generation config keys: dict_keys(['project_root', 'data_root', 'output', 'generation', 'simulation', 'parameter_sampler', 'detectors', 'signal_processor', 'label_transformer'])
Training config keys: dict_keys(['project_root', 'data_root', 'dataset', 'model', 'training', 'outputs'])
Training parameters: dict_keys(['n_detectors', 'n_outputs', 'embedding_dim', 'residual_channels', 'dilations', 'residual_kernel_size', 'dropout_conv', 'dropout_dense', 'num_groups'])


In [3]:
detectors = gen_cfg["detectors"]

fs = 4096
duration = gen_cfg["simulation"]["duration"]
n_samples = int(duration * fs)

context_start = gen_cfg["simulation"]["processing_context_start_samples"]
context_end = gen_cfg["simulation"]["processing_context_end_samples"]
processing_length = n_samples + context_start + context_end

signal_processor_cfg = gen_cfg["signal_processor"]

print("Detector order:", detectors)
print("Sampling frequency:", fs)
print("Final duration:", duration)
print("Final samples:", n_samples)
print("Processing context start samples:", context_start)
print("Processing context end samples:", context_end)
print("Processing input length:", processing_length)
print("Processing input duration:", processing_length / fs)

print("\nSignal processor:")
for k, v in signal_processor_cfg.items():
    print(f"  {k}: {v}")

Detector order: ['H1', 'L1', 'V1']
Sampling frequency: 4096
Final duration: 4.0
Final samples: 16384
Processing context start samples: 1664
Processing context end samples: 1664
Processing input length: 19712
Processing input duration: 4.8125

Signal processor:
  whitening_method: psd
  apply_highpass: True
  apply_lowpass: True
  apply_standardization: False
  output_mode: crop_to_config
  whitening_low_frequency_cutoff: 30.0
  whitening_max_filter_duration: 0.5
  whitening_trunc_method: hann
  highpass_frequency: 30.0
  lowpass_frequency: 512.0
  fir_order: 256
  fir_beta: 5.0
  remove_corrupted: True


In [4]:
### SANITY CHECKS to assure that the configuration parameters are consistent with the expected values

assert detectors == ["H1", "L1", "V1"], detectors
assert fs == 4096
assert n_samples == 16384
assert processing_length == 19712

assert signal_processor_cfg["whitening_method"] == "psd"
assert signal_processor_cfg["apply_highpass"] is True
assert signal_processor_cfg["apply_lowpass"] is True
assert signal_processor_cfg["apply_standardization"] is False
assert signal_processor_cfg["highpass_frequency"] == 30.0
assert signal_processor_cfg["lowpass_frequency"] == 512.0

print("Input contract validated.")

Input contract validated.


In [5]:
from src.models.network import SimpleCNN_ResidualDilated

# Getting model info
model_cfg = train_cfg["model"]

print(model_cfg["class_name"])
print(model_cfg["kwargs"])

model = SimpleCNN_ResidualDilated(**model_cfg["kwargs"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")

SimpleCNN_ResidualDilated
{'n_detectors': 3, 'n_outputs': 3, 'embedding_dim': 64, 'residual_channels': 64, 'dilations': [1, 2, 4], 'residual_kernel_size': 7, 'dropout_conv': 0.05, 'dropout_dense': 0.1, 'num_groups': 8}
SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=Fals

In [6]:
checkpoint_tag = train_cfg["outputs"]["checkpoint_tag"]
print("Checkpoint tag:", checkpoint_tag)

candidate_checkpoints = sorted(MODEL_DIR.rglob(f"*{checkpoint_tag}*"))
for p in candidate_checkpoints[:20]:
    print(p)

print("Number of candidates:", len(candidate_checkpoints))

Checkpoint tag: M08_resdilated_emb64_d124
/data/vserrano/cbc_pe_data/models/checkpoints/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_MSELoss_seed124_checkpoint.pt
Number of candidates: 1


In [7]:
checkpoint_path = candidate_checkpoints[-1]  # Load the last checkpoint

ckpt = torch.load(checkpoint_path, map_location="cpu")
print(ckpt.keys())

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'best_val_loss', 'y_mean', 'y_std', 'model_config', 'training_config', 'elapsed_seconds', 'history'])


SimpleCNN_ResidualDilated(
  (block1): ConvBlock(
    (conv): Conv1d(3, 16, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 16, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block2): ConvBlock(
    (conv): Conv1d(16, 32, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 32, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (block3): ConvBlock(
    (conv): Conv1d(32, 64, kernel_size=(16,), stride=(2,), padding=(8,))
    (group_norm): GroupNorm(8, 64, eps=1e-05, affine=True)
    (activation): LeakyReLU(negative_slope=0.01)
    (dropout): Dropout(p=0.05, inplace=False)
  )
  (residual_blocks): Sequential(
    (0): ResidualDilatedBlock(
      (conv1): Conv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
      (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
      (activation

In [8]:
# ------------------------------------------------------------
# 6. Label normalization statistics
# ------------------------------------------------------------

y_mean = np.asarray(ckpt["y_mean"], dtype=np.float64)
y_std = np.asarray(ckpt["y_std"], dtype=np.float64)

label_names = ["chirp_mass", "total_mass", "chi_eff"]

print("Label names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)

assert y_mean.shape == (3,)
assert y_std.shape == (3,)
assert np.all(np.isfinite(y_mean))
assert np.all(np.isfinite(y_std))
assert np.all(y_std > 0)

print("Label statistics validated.")

Label names: ['chirp_mass', 'total_mass', 'chi_eff']
y_mean: [3.74526215e+01 9.49739685e+01 1.02269521e-03]
y_std: [16.48472214 34.65016556  0.44085518]
Label statistics validated.


In [9]:
def inverse_standardize(y_std_space):
    """
    Convert standardized labels/predictions to physical units:
    [chirp_mass, total_mass, chi_eff].
    """
    y_std_space = np.asarray(y_std_space, dtype=np.float64)
    return y_std_space * y_std + y_mean


def standardize(y_phys):
    """
    Convert physical labels to standardized training space.
    """
    y_phys = np.asarray(y_phys, dtype=np.float64)
    return (y_phys - y_mean) / y_std

test_std = np.zeros((1, 3))
test_phys = inverse_standardize(test_std)

print("Zero standardized corresponds to physical mean:")
for name, value in zip(label_names, test_phys[0]):
    print(f"{name}: {value:.6g}")

Zero standardized corresponds to physical mean:
chirp_mass: 37.4526
total_mass: 94.974
chi_eff: 0.0010227


In [10]:
def predict_m08(model, X, device="cpu"):
    """
    Run M08 inference.

    Parameters
    ----------
    model : torch.nn.Module
        Loaded M08 model.
    X : np.ndarray
        Shape (n_events, 3, 16384) or (3, 16384).

    Returns
    -------
    pred_std : np.ndarray
        Standardized predictions, shape (n_events, 3).
    pred_phys : np.ndarray
        Physical predictions, shape (n_events, 3).
    emb : np.ndarray
        Embeddings, shape (n_events, 64).
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 2:
        X = X[None, :, :]

    if X.shape[1:] != (3, 16384):
        raise ValueError(f"Expected X shape (N, 3, 16384), got {X.shape}")

    model = model.to(device)
    model.eval()

    x_tensor = torch.from_numpy(X).to(device)

    with torch.no_grad():
        pred_std_t, emb_t = model(x_tensor, return_embedding=True)

    pred_std = pred_std_t.cpu().numpy()
    emb = emb_t.cpu().numpy()
    pred_phys = inverse_standardize(pred_std)

    return pred_std, pred_phys, emb

In [11]:
# ------------------------------------------------------------
# Load M08 prediction/embedding file
# ------------------------------------------------------------

DATASET_ID = train_cfg["dataset"]["dataset_id"]

prediction_candidates = sorted(
    RESULTS_DIR.rglob(
        f"{DATASET_ID}_SimpleCNN_ResidualDilated_M08*predictions_embeddings*.npz"
    )
)

print("Prediction/embedding candidates:")
for p in prediction_candidates:
    print(" ", p)

print("Number of candidates:", len(prediction_candidates))

assert len(prediction_candidates) >= 1, "No M08 prediction/embedding file found."

M08_PRED_PATH = prediction_candidates[0]
print("Selected:", M08_PRED_PATH)

Prediction/embedding candidates:
  /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_MSELoss_seed124_val_cal_test_predictions_embeddings.npz
  /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08a_residual_emb64_d111_MSELoss_seed123_val_cal_test_predictions_embeddings.npz
Number of candidates: 2
Selected: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000_SimpleCNN_ResidualDilated_M08_resdilated_emb64_d124_MSELoss_seed124_val_cal_test_predictions_embeddings.npz


In [12]:
m08_data = np.load(M08_PRED_PATH, allow_pickle=True)

print("Available keys:")
for key in sorted(m08_data.files):
    value = m08_data[key]
    print(f"{key:25s} shape={value.shape} dtype={value.dtype}")

Available keys:
available_splits          shape=(3,) dtype=<U4
checkpoint_file           shape=() dtype=<U174
dataset_path              shape=() dtype=<U86
emb_cal                   shape=(30000, 64) dtype=float32
emb_test                  shape=(30000, 64) dtype=float32
emb_val                   shape=(40000, 64) dtype=float32
idx_cal                   shape=(30000,) dtype=int64
idx_test                  shape=(30000,) dtype=int64
idx_val                   shape=(40000,) dtype=int64
label_names               shape=(3,) dtype=<U10
label_stats_path          shape=() dtype=<U158
model_config              shape=() dtype=object
pred_cal                  shape=(30000, 3) dtype=float32
pred_test                 shape=(30000, 3) dtype=float32
pred_val                  shape=(40000, 3) dtype=float32
split_path                shape=() dtype=<U142
y_cal                     shape=(30000, 3) dtype=float32
y_mean                    shape=(3,) dtype=float32
y_std                     shape=(3,) dtype

In [13]:
SPLITS = {}

for split in ["val", "cal", "test"]:
    required = [f"pred_{split}", f"y_{split}", f"emb_{split}"]

    missing = [key for key in required if key not in m08_data.files]
    if missing:
        print(f"Skipping split={split}, missing keys:", missing)
        continue

    SPLITS[split] = {
        "pred": np.asarray(m08_data[f"pred_{split}"], dtype=np.float64),
        "y": np.asarray(m08_data[f"y_{split}"], dtype=np.float64),
        "emb": np.asarray(m08_data[f"emb_{split}"], dtype=np.float64),
    }

    idx_key = f"idx_{split}"
    if idx_key in m08_data.files:
        SPLITS[split]["idx"] = np.asarray(m08_data[idx_key])

for split, data in SPLITS.items():
    print(f"\nSplit: {split}")
    for key, value in data.items():
        print(f"  {key:5s}: {value.shape}")


Split: val
  pred : (40000, 3)
  y    : (40000, 3)
  emb  : (40000, 64)
  idx  : (40000,)

Split: cal
  pred : (30000, 3)
  y    : (30000, 3)
  emb  : (30000, 64)
  idx  : (30000,)

Split: test
  pred : (30000, 3)
  y    : (30000, 3)
  emb  : (30000, 64)
  idx  : (30000,)


In [14]:
# ------------------------------------------------------------
# 9. Load selected Mondrian configurations
# ------------------------------------------------------------

MONDRIAN_DIR = RESULTS_DIR / "mondrian_M08_final_baseline"

print("MONDRIAN_DIR:", MONDRIAN_DIR)
print("Exists:", MONDRIAN_DIR.exists())

for p in sorted(MONDRIAN_DIR.glob("*")):
    print(p.name)

MONDRIAN_DIR: /data/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/mondrian_M08_final_baseline
Exists: True
conditional_metrics.csv
figures_academic
global_conformal.csv
global_point_metrics.csv
grid_aggregate_summary.csv
mondrian_summary_all.csv
point_metrics.csv
selected_configurations.csv
selected_systems_report.txt
selected_systems_summary.csv
test_metadata.csv
top_candidates.csv


In [15]:
import pandas as pd

selected_config_path = MONDRIAN_DIR / "selected_configurations.csv"
selected_systems_path = MONDRIAN_DIR / "selected_systems_summary.csv"

selection_df = pd.read_csv(selected_config_path)
selected_systems_df = pd.read_csv(selected_systems_path)

display(selection_df)
display(selected_systems_df)

,taxonomy_mode,interval_mode,n_bins,label,label_index,global_coverage,global_miscoverage,global_undercoverage_pvalue,global_mean_width_std,global_median_width_std,...,global_2sigma_low,global_2sigma_high,global_3sigma_low,global_3sigma_high,global_within_2sigma,min_width_for_label,width_limit,relative_width_excess,selection_policy,final_policy
0,prediction,asymmetric,24,chirp_mass,0,0.902767,0.097233,0.946419,0.905068,0.947766,...,0.896536,0.903464,0.894804,0.905196,True,15.439307,15.748093,0.01194,conservative_zero_under_bins_2sigma,conservative
1,difficulty,symmetric,24,chirp_mass,0,0.901567,0.098433,0.819569,0.896450,0.870898,...,0.896536,0.903464,0.894804,0.905196,True,14.356512,14.643642,0.00000,efficient_local_validity_tolerant,efficient
2,prediction,asymmetric,16,total_mass,1,0.901067,0.098933,0.733653,0.846922,0.805739,...,0.896536,0.903464,0.894804,0.905196,True,27.919006,28.477386,0.00000,conservative_zero_under_bins_2sigma,conservative
3,prediction,asymmetric,32,total_mass,1,0.901500,0.098500,0.809228,0.850464,0.787492,...,0.896536,0.903464,0.894804,0.905196,True,27.286712,27.832446,0.00000,efficient_local_validity_tolerant,efficient
4,difficulty,symmetric,6,chi_eff,2,0.898400,0.101600,0.180210,1.345089,1.307212,...,0.896536,0.903464,0.894804,0.905196,True,0.576291,0.587817,0.00000,conservative_zero_under_bins_2sigma,conservative
5,difficulty,symmetric,12,chi_eff,2,0.899067,0.100933,0.297681,1.344295,1.273241,...,0.896536,0.903464,0.894804,0.905196,True,0.561315,0.572541,0.00000,efficient_local_validity_tolerant,efficient


,final_policy,label,taxonomy_mode,interval_mode,n_bins,coverage,median_width_phys,q90_width_phys,q95_width_phys,min_coverage_per_bin,n_bins_under_2sigma,max_undercoverage_gap,tail_miss_imbalance
0,conservative,chirp_mass,prediction,asymmetric,24,0.902767,15.623656,21.191378,21.730857,0.886381,0,0.013619,0.000233
1,conservative,total_mass,prediction,asymmetric,16,0.901067,27.919006,37.565449,38.805998,0.887652,0,0.012348,0.001933
2,conservative,chi_eff,difficulty,symmetric,6,0.898400,0.576291,0.812271,0.812271,0.892167,0,0.007833,0.002067
3,efficient,chirp_mass,difficulty,symmetric,24,0.901567,14.356512,23.366214,24.942036,0.873544,1,0.026456,0.002367
4,efficient,total_mass,prediction,asymmetric,32,0.901500,27.286712,37.805153,38.552800,0.874737,2,0.025263,0.001833
5,efficient,chi_eff,difficulty,symmetric,12,0.899067,0.561315,0.721185,0.894796,0.882232,1,0.017768,0.002133


In [16]:


MONDRIAN_POLICY = "conservative"

selected_policy_df = selected_systems_df[
    selected_systems_df["final_policy"] == MONDRIAN_POLICY
].copy()

display(selected_policy_df)

assert len(selected_policy_df) == 3
assert set(selected_policy_df["label"]) == set(label_names)

selected_configs = {}

for _, row in selected_policy_df.iterrows():
    label = row["label"]
    selected_configs[label] = {
        "label": label,
        "label_index": label_names.index(label),
        "taxonomy_mode": row["taxonomy_mode"],
        "interval_mode": row["interval_mode"],
        "n_bins": int(row["n_bins"]),
        "coverage": float(row["coverage"]),
        "median_width_phys": float(row["median_width_phys"]),
        "q90_width_phys": float(row["q90_width_phys"]),
        "q95_width_phys": float(row["q95_width_phys"]),
        "min_coverage_per_bin": float(row["min_coverage_per_bin"]),
    }

selected_configs

,final_policy,label,taxonomy_mode,interval_mode,n_bins,coverage,median_width_phys,q90_width_phys,q95_width_phys,min_coverage_per_bin,n_bins_under_2sigma,max_undercoverage_gap,tail_miss_imbalance
0,conservative,chirp_mass,prediction,asymmetric,24,0.902767,15.623656,21.191378,21.730857,0.886381,0,0.013619,0.000233
1,conservative,total_mass,prediction,asymmetric,16,0.901067,27.919006,37.565449,38.805998,0.887652,0,0.012348,0.001933
2,conservative,chi_eff,difficulty,symmetric,6,0.898400,0.576291,0.812271,0.812271,0.892167,0,0.007833,0.002067


{'chirp_mass': {'label': 'chirp_mass',
  'label_index': 0,
  'taxonomy_mode': 'prediction',
  'interval_mode': 'asymmetric',
  'n_bins': 24,
  'coverage': 0.9027666666666668,
  'median_width_phys': 15.623655575492135,
  'q90_width_phys': 21.191378383142364,
  'q95_width_phys': 21.7308572319746,
  'min_coverage_per_bin': 0.8863813229571984},
 'total_mass': {'label': 'total_mass',
  'label_index': 1,
  'taxonomy_mode': 'prediction',
  'interval_mode': 'asymmetric',
  'n_bins': 16,
  'coverage': 0.9010666666666668,
  'median_width_phys': 27.919005509654284,
  'q90_width_phys': 37.56544883505717,
  'q95_width_phys': 38.80599842588231,
  'min_coverage_per_bin': 0.8876523582405935},
 'chi_eff': {'label': 'chi_eff',
  'label_index': 2,
  'taxonomy_mode': 'difficulty',
  'interval_mode': 'symmetric',
  'n_bins': 6,
  'coverage': 0.8984,
  'median_width_phys': 0.5762910812388933,
  'q90_width_phys': 0.8122714828905009,
  'q95_width_phys': 0.8122714828905009,
  'min_coverage_per_bin': 0.89216683

In [17]:
from src.conformal.binning import QuantileBinner, BinGrouper
from src.conformal.calibration import ConformalIntervalCalibrator
from src.conformal.difficulty import DifficultyEstimator

In [18]:
from dataclasses import dataclass

@dataclass
class SelectedTargetMondrianSystem:
    label: str
    label_index: int
    taxonomy_mode: str
    interval_mode: str
    n_bins: int
    binner: object
    calibrator: object
    difficulty_model: object | None
    intervals_std: np.ndarray
    bin_indices_cal: np.ndarray
    binning_scores_cal: np.ndarray

In [19]:
def fit_selected_target_mondrian_system(
    *,
    label: str,
    label_index: int,
    taxonomy_mode: str,
    interval_mode: str,
    n_bins: int,
    pred_cal: np.ndarray,
    y_cal: np.ndarray,
    emb_cal: np.ndarray,
    confidence_level: float = 0.90,
    n_neighbors: int = 5,
    apply_jitter: bool = False,
    jitter_variation: float = 1e-10,
    min_samples_per_bin: int = 10,
):
    """
    Fit one selected Mondrian conformal system for one target.

    Everything is fitted using calibration data only.

    Parameters
    ----------
    label_index:
        0 -> chirp_mass
        1 -> total_mass
        2 -> chi_eff

    pred_cal, y_cal:
        Standardized calibration predictions and labels.

    emb_cal:
        Calibration embeddings from M08.

    Returns
    -------
    SelectedTargetMondrianSystem
    """
    pred_cal = np.asarray(pred_cal, dtype=np.float64)
    y_cal = np.asarray(y_cal, dtype=np.float64)
    emb_cal = np.asarray(emb_cal, dtype=np.float64)

    assert pred_cal.shape == y_cal.shape
    assert pred_cal.ndim == 2
    assert emb_cal.ndim == 2
    assert pred_cal.shape[0] == emb_cal.shape[0]

    # Residual definition used by your conformal code:
    # residual = y_cal - pred_cal
    residuals_cal_all = y_cal - pred_cal
    residuals_cal_target = residuals_cal_all[:, [label_index]]

    # ------------------------------------------------------------
    # 1. Build binning scores
    # ------------------------------------------------------------
    if taxonomy_mode == "prediction":
        # Prediction taxonomy:
        # score = model prediction for that target.
        binning_scores_cal = pred_cal[:, [label_index]]
        difficulty_model = None

    elif taxonomy_mode == "difficulty":
        # Difficulty taxonomy:
        # fit kNN in embedding space using calibration residuals.
        #
        # Important:
        # We pass residuals for all labels, because your DifficultyEstimator
        # computes one difficulty score per label.
        difficulty_model = DifficultyEstimator(n_neighbors=n_neighbors)
        difficulty_model.calibrate_estimator(
            cal_embedding=emb_cal,
            cal_residuals=residuals_cal_all,
        )

        difficulty_scores_cal_all = difficulty_model.compute_calibration_difficulty()
        binning_scores_cal = difficulty_scores_cal_all[:, [label_index]]

    else:
        raise ValueError(f"Unknown taxonomy_mode: {taxonomy_mode}")

    # ------------------------------------------------------------
    # 2. Fit quantile bin edges using calibration scores
    # ------------------------------------------------------------
    binner = QuantileBinner(
        n_bins=n_bins,
        apply_jitter=apply_jitter,
        jitter_variation=jitter_variation,
    )

    bin_indices_cal = binner.bin_edges_and_indices(binning_scores_cal)

    # ------------------------------------------------------------
    # 3. Group calibration residuals by bin
    # ------------------------------------------------------------
    grouper = BinGrouper()
    grouped_residuals = grouper.group_by_bin(
        residuals=residuals_cal_target,
        bin_indices=bin_indices_cal,
        n_bins=n_bins,
    )

    # ------------------------------------------------------------
    # 4. Fit conformal offsets
    # ------------------------------------------------------------
    calibrator = ConformalIntervalCalibrator(
        confidence_level=confidence_level,
        interval_mode=interval_mode,
        min_samples_per_bin=min_samples_per_bin,
    )
    calibrator.fit(grouped_residuals)

    intervals_std = calibrator.intervals_

    # Since this is one target only, shape should be:
    # (1, n_bins, 2)
    assert intervals_std.shape == (1, n_bins, 2)

    return SelectedTargetMondrianSystem(
        label=label,
        label_index=label_index,
        taxonomy_mode=taxonomy_mode,
        interval_mode=interval_mode,
        n_bins=n_bins,
        binner=binner,
        calibrator=calibrator,
        difficulty_model=difficulty_model,
        intervals_std=intervals_std,
        bin_indices_cal=bin_indices_cal,
        binning_scores_cal=binning_scores_cal,
    )

In [20]:
CONFIDENCE_LEVEL = 0.90
N_NEIGHBORS = 5

# Usa estos nombres tal como ya los tenías
pred_cal = SPLITS["cal"]["pred"]
y_cal = SPLITS["cal"]["y"]
emb_cal = SPLITS["cal"]["emb"]

selected_mondrian_systems = {}

for label, cfg in selected_configs.items():
    print(f"Fitting {label}: {cfg}")

    system = fit_selected_target_mondrian_system(
        label=label,
        label_index=cfg["label_index"],
        taxonomy_mode=cfg["taxonomy_mode"],
        interval_mode=cfg["interval_mode"],
        n_bins=cfg["n_bins"],
        pred_cal=pred_cal,
        y_cal=y_cal,
        emb_cal=emb_cal,
        confidence_level=CONFIDENCE_LEVEL,
        n_neighbors=N_NEIGHBORS,
        apply_jitter=False,
        min_samples_per_bin=10,
    )

    selected_mondrian_systems[label] = system

print("\nFitted systems:")
for label, system in selected_mondrian_systems.items():
    print(
        label,
        "| taxonomy:", system.taxonomy_mode,
        "| interval:", system.interval_mode,
        "| n_bins:", system.n_bins,
        "| intervals shape:", system.intervals_std.shape,
    )

Fitting chirp_mass: {'label': 'chirp_mass', 'label_index': 0, 'taxonomy_mode': 'prediction', 'interval_mode': 'asymmetric', 'n_bins': 24, 'coverage': 0.9027666666666668, 'median_width_phys': 15.623655575492135, 'q90_width_phys': 21.191378383142364, 'q95_width_phys': 21.7308572319746, 'min_coverage_per_bin': 0.8863813229571984}
Fitting total_mass: {'label': 'total_mass', 'label_index': 1, 'taxonomy_mode': 'prediction', 'interval_mode': 'asymmetric', 'n_bins': 16, 'coverage': 0.9010666666666668, 'median_width_phys': 27.919005509654284, 'q90_width_phys': 37.56544883505717, 'q95_width_phys': 38.80599842588231, 'min_coverage_per_bin': 0.8876523582405935}
Fitting chi_eff: {'label': 'chi_eff', 'label_index': 2, 'taxonomy_mode': 'difficulty', 'interval_mode': 'symmetric', 'n_bins': 6, 'coverage': 0.8984, 'median_width_phys': 0.5762910812388933, 'q90_width_phys': 0.8122714828905009, 'q95_width_phys': 0.8122714828905009, 'min_coverage_per_bin': 0.8921668362156663}

Fitted systems:
chirp_mass | t

In [21]:
def apply_selected_mondrian_systems(
    *,
    pred_std: np.ndarray,
    emb: np.ndarray,
    selected_mondrian_systems: dict,
):
    """
    Apply selected per-target Mondrian systems to new predictions.

    Parameters
    ----------
    pred_std:
        Standardized M08 predictions, shape (N, 3).

    emb:
        M08 embeddings, shape (N, 64).

    Returns
    -------
    lower_std, upper_std:
        Standardized conformal interval bounds, shape (N, 3).

    bin_indices:
        Assigned bin per sample/target, shape (N, 3).

    binning_scores:
        Binning score per sample/target, shape (N, 3).
    """
    pred_std = np.asarray(pred_std, dtype=np.float64)
    emb = np.asarray(emb, dtype=np.float64)

    if pred_std.ndim != 2 or pred_std.shape[1] != 3:
        raise ValueError(f"Expected pred_std shape (N, 3), got {pred_std.shape}")

    if emb.ndim != 2:
        raise ValueError(f"Expected emb shape (N, embedding_dim), got {emb.shape}")

    if emb.shape[0] != pred_std.shape[0]:
        raise ValueError("pred_std and emb must have the same number of samples.")

    n_samples = pred_std.shape[0]

    lower_std = np.empty_like(pred_std)
    upper_std = np.empty_like(pred_std)
    bin_indices = np.empty_like(pred_std, dtype=int)
    binning_scores = np.empty_like(pred_std)

    for label, system in selected_mondrian_systems.items():
        j = system.label_index

        # ------------------------------------------------------------
        # 1. Compute target binning score
        # ------------------------------------------------------------
        if system.taxonomy_mode == "prediction":
            scores_new = pred_std[:, [j]]

        elif system.taxonomy_mode == "difficulty":
            if system.difficulty_model is None:
                raise ValueError(f"{label} requires a fitted difficulty model.")

            difficulty_scores_all = system.difficulty_model.compute_target_difficulty(
                target_embedding=emb
            )
            scores_new = difficulty_scores_all[:, [j]]

        else:
            raise ValueError(f"Unknown taxonomy_mode: {system.taxonomy_mode}")

        # ------------------------------------------------------------
        # 2. Assign bin
        # ------------------------------------------------------------
        bins_new = system.binner.get_bin_indices(scores_new)

        # Shape checks: one target only
        assert bins_new.shape == (n_samples, 1)

        # ------------------------------------------------------------
        # 3. Select calibrated offsets for assigned bin
        # ------------------------------------------------------------
        lower_offsets = system.intervals_std[0, bins_new[:, 0], 0]
        upper_offsets = system.intervals_std[0, bins_new[:, 0], 1]

        lower_std[:, j] = pred_std[:, j] + lower_offsets
        upper_std[:, j] = pred_std[:, j] + upper_offsets

        bin_indices[:, j] = bins_new[:, 0]
        binning_scores[:, j] = scores_new[:, 0]

    if np.any(lower_std > upper_std):
        raise ValueError("Some intervals have lower > upper.")

    return lower_std, upper_std, bin_indices, binning_scores

In [22]:
def intervals_std_to_phys(lower_std, upper_std):
    """
    Convert standardized interval bounds to physical units.
    """
    lower_std = np.asarray(lower_std, dtype=np.float64)
    upper_std = np.asarray(upper_std, dtype=np.float64)

    lower_phys = inverse_standardize(lower_std)
    upper_phys = inverse_standardize(upper_std)

    return lower_phys, upper_phys

In [23]:
pred_test = SPLITS["test"]["pred"]
y_test = SPLITS["test"]["y"]
emb_test = SPLITS["test"]["emb"]

lower_test_std, upper_test_std, bins_test, scores_test = apply_selected_mondrian_systems(
    pred_std=pred_test,
    emb=emb_test,
    selected_mondrian_systems=selected_mondrian_systems,
)

print("lower_test_std:", lower_test_std.shape)
print("upper_test_std:", upper_test_std.shape)
print("bins_test:", bins_test.shape)

assert lower_test_std.shape == pred_test.shape
assert upper_test_std.shape == pred_test.shape
assert bins_test.shape == pred_test.shape

lower_test_std: (30000, 3)
upper_test_std: (30000, 3)
bins_test: (30000, 3)


In [24]:
covered = (lower_test_std <= y_test) & (y_test <= upper_test_std)
coverage = covered.mean(axis=0)

width_std = upper_test_std - lower_test_std
median_width_std = np.median(width_std, axis=0)

print("Global coverage:")
for j, name in enumerate(label_names):
    print(f"  {name:12s}: {coverage[j]:.6f}")

print("\nMedian width [standardized]:")
for j, name in enumerate(label_names):
    print(f"  {name:12s}: {median_width_std[j]:.6f}")

Global coverage:
  chirp_mass  : 0.899233
  total_mass  : 0.903033
  chi_eff     : 0.903300

Median width [standardized]:
  chirp_mass  : 0.924717
  total_mass  : 0.794647
  chi_eff     : 1.277579


In [25]:
lower_test_phys, upper_test_phys = intervals_std_to_phys(
    lower_test_std,
    upper_test_std,
)

y_test_phys = inverse_standardize(y_test)
pred_test_phys = inverse_standardize(pred_test)

width_phys = upper_test_phys - lower_test_phys
median_width_phys = np.median(width_phys, axis=0)
q90_width_phys = np.quantile(width_phys, 0.90, axis=0)
q95_width_phys = np.quantile(width_phys, 0.95, axis=0)

print("Physical width summary:")
for j, name in enumerate(label_names):
    print(
        f"{name:12s} | "
        f"median={median_width_phys[j]:.6f} | "
        f"q90={q90_width_phys[j]:.6f} | "
        f"q95={q95_width_phys[j]:.6f}"
    )

Physical width summary:
chirp_mass   | median=15.243710 | q90=21.185392 | q95=21.612253
total_mass   | median=27.534664 | q90=38.627888 | q95=38.888468
chi_eff      | median=0.563227 | q90=0.792811 | q95=0.792811


In [26]:
validation_rows = []

for j, name in enumerate(label_names):
    validation_rows.append({
        "label": name,
        "coverage_reconstructed": coverage[j],
        "median_width_phys_reconstructed": median_width_phys[j],
        "q90_width_phys_reconstructed": q90_width_phys[j],
        "q95_width_phys_reconstructed": q95_width_phys[j],
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

display(selected_policy_df[
    [
        "label",
        "coverage",
        "median_width_phys",
        "q90_width_phys",
        "q95_width_phys",
        "min_coverage_per_bin",
    ]
])

,label,coverage_reconstructed,median_width_phys_reconstructed,q90_width_phys_reconstructed,q95_width_phys_reconstructed
0,chirp_mass,0.899233,15.243710,21.185392,21.612253
1,total_mass,0.903033,27.534664,38.627888,38.888468
2,chi_eff,0.903300,0.563227,0.792811,0.792811


,label,coverage,median_width_phys,q90_width_phys,q95_width_phys,min_coverage_per_bin
0,chirp_mass,0.902767,15.623656,21.191378,21.730857,0.886381
1,total_mass,0.901067,27.919006,37.565449,38.805998,0.887652
2,chi_eff,0.898400,0.576291,0.812271,0.812271,0.892167


In [27]:
from src.conformal.pipeline import run_mondrian_regression

def rerun_original_style_for_selected_row(cfg):
    label = cfg["label"]
    j = cfg["label_index"]

    kwargs = dict(
        pred_cal=SPLITS["cal"]["pred"],
        pred_test=SPLITS["test"]["pred"],
        y_cal=SPLITS["cal"]["y"],
        y_test=SPLITS["test"]["y"],
        n_bins=cfg["n_bins"],
        confidence_level=0.90,
        apply_jitter=False,
        interval_mode=cfg["interval_mode"],
        taxonomy_mode=cfg["taxonomy_mode"],
        min_samples_per_bin=10,
        tolerance_sigmas=(1, 2, 3),
    )

    if cfg["taxonomy_mode"] == "difficulty":
        kwargs["cal_embedding"] = SPLITS["cal"]["emb"]
        kwargs["target_embedding"] = SPLITS["test"]["emb"]
        kwargs["n_neighbors"] = 5

    result = run_mondrian_regression(**kwargs)

    lower_phys = inverse_standardize(result.lower)
    upper_phys = inverse_standardize(result.upper)

    widths_phys = upper_phys - lower_phys

    return {
        "label": label,
        "coverage_original_style": result.metrics["global_coverage"][j],
        "median_width_phys_original_style": np.median(widths_phys[:, j]),
        "q90_width_phys_original_style": np.quantile(widths_phys[:, j], 0.90),
        "q95_width_phys_original_style": np.quantile(widths_phys[:, j], 0.95),
    }


original_style_rows = []

for label, cfg in selected_configs.items():
    original_style_rows.append(rerun_original_style_for_selected_row(cfg))

original_style_df = pd.DataFrame(original_style_rows)
display(original_style_df)

display(validation_df)
display(selected_policy_df[
    [
        "label",
        "coverage",
        "median_width_phys",
        "q90_width_phys",
        "q95_width_phys",
    ]
])

,label,coverage_original_style,median_width_phys_original_style,q90_width_phys_original_style,q95_width_phys_original_style
0,chirp_mass,0.899233,15.243710,21.185392,21.612253
1,total_mass,0.903033,27.534664,38.627888,38.888468
2,chi_eff,0.903300,0.563227,0.792811,0.792811


,label,coverage_reconstructed,median_width_phys_reconstructed,q90_width_phys_reconstructed,q95_width_phys_reconstructed
0,chirp_mass,0.899233,15.243710,21.185392,21.612253
1,total_mass,0.903033,27.534664,38.627888,38.888468
2,chi_eff,0.903300,0.563227,0.792811,0.792811


,label,coverage,median_width_phys,q90_width_phys,q95_width_phys
0,chirp_mass,0.902767,15.623656,21.191378,21.730857
1,total_mass,0.901067,27.919006,37.565449,38.805998
2,chi_eff,0.898400,0.576291,0.812271,0.812271


In [28]:
comparison_df = (
    validation_df
    .merge(
        selected_policy_df[
            ["label", "coverage", "median_width_phys", "q90_width_phys", "q95_width_phys"]
        ],
        on="label",
        how="left"
    )
)

comparison_df["delta_coverage"] = (
    comparison_df["coverage_reconstructed"] - comparison_df["coverage"]
)

comparison_df["delta_median_width_phys"] = (
    comparison_df["median_width_phys_reconstructed"] - comparison_df["median_width_phys"]
)

comparison_df["rel_delta_median_width"] = (
    comparison_df["delta_median_width_phys"] / comparison_df["median_width_phys"]
)

display(comparison_df)

,label,coverage_reconstructed,median_width_phys_reconstructed,q90_width_phys_reconstructed,q95_width_phys_reconstructed,coverage,median_width_phys,q90_width_phys,q95_width_phys,delta_coverage,delta_median_width_phys,rel_delta_median_width
0,chirp_mass,0.899233,15.243710,21.185392,21.612253,0.902767,15.623656,21.191378,21.730857,-0.003533,-0.379945,-0.024319
1,total_mass,0.903033,27.534664,38.627888,38.888468,0.901067,27.919006,37.565449,38.805998,0.001967,-0.384342,-0.013766
2,chi_eff,0.903300,0.563227,0.792811,0.792811,0.898400,0.576291,0.812271,0.812271,0.004900,-0.013064,-0.022668


The reconstructed metrics are recomputed from the calibration/test arrays loaded in this notebook. The selected metrics are read from the previous Mondrian summary CSV and are used only to identify the selected hyperparameter configuration. For real-event inference, the reconstructed calibrators are the operative objects, because they are fitted in the current notebook from the loaded calibration arrays.

## 13. Real-event data loading

We load long GWOSC strain files for one HLV event. Long files are needed because real-data PSD whitening requires an off-source PSD estimate. This replaces the synthetic training step `NoiseModel.get_psd(...)`.

In [29]:
from pathlib import Path
import urllib.request
import h5py
from pycbc.types import TimeSeries as PyCBCTimeSeries

event_name = "GW170814"
event_time = 1186741861.5

REAL_DETECTORS = ["H1", "L1", "V1"]

GWOSC_URLS_4096 = {
    "H1": "https://gwosc.org/archive/data/O2_4KHZ_R1/1185939456/H-H1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5",
    "L1": "https://gwosc.org/archive/data/O2_4KHZ_R1/1185939456/L-L1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5",
    "V1": "https://gwosc.org/archive/data/O2_4KHZ_R1/1185939456/V-V1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5",
}

GWOSC_CACHE_DIR = Path("gwosc_cache")
GWOSC_CACHE_DIR.mkdir(exist_ok=True)

/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal


In [30]:
def download_if_needed(url, cache_dir=GWOSC_CACHE_DIR):
    filename = url.split("/")[-1]
    local_path = cache_dir / filename

    if not local_path.exists():
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, local_path)
    else:
        print(f"Using cached file: {filename}")

    return local_path


def read_gwosc_hdf5_as_pycbc_timeseries(path):
    with h5py.File(path, "r") as f:
        strain = f["strain"]["Strain"][:].astype(np.float64)
        gps_start = float(f["meta"]["GPSstart"][()])
        duration = float(f["meta"]["Duration"][()])
        delta_t = duration / len(strain)

    return PyCBCTimeSeries(
        strain,
        delta_t=delta_t,
        epoch=gps_start,
    )

In [31]:
raw_strains_4096 = {}

for ifo, url in GWOSC_URLS_4096.items():
    print(f"\nLoading {ifo}")

    local_path = download_if_needed(url)
    ts = read_gwosc_hdf5_as_pycbc_timeseries(local_path)

    raw_strains_4096[ifo] = ts

    print(
        ifo,
        "len:", len(ts),
        "duration:", float(ts.duration),
        "sample_rate:", float(ts.sample_rate),
        "start:", float(ts.start_time),
        "end:", float(ts.end_time),
    )

    assert abs(float(ts.sample_rate) - 4096.0) < 1e-6
    assert float(ts.start_time) <= event_time <= float(ts.end_time)


Loading H1
Using cached file: H-H1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
H1 len: 16777216 duration: 4096.0 sample_rate: 4096.0 start: 1186738176.0 end: 1186742272.0

Loading L1
Using cached file: L-L1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
L1 len: 16777216 duration: 4096.0 sample_rate: 4096.0 start: 1186738176.0 end: 1186742272.0

Loading V1
Using cached file: V-V1_GWOSC_O2_4KHZ_R1-1186738176-4096.hdf5
V1 len: 16777216 duration: 4096.0 sample_rate: 4096.0 start: 1186738176.0 end: 1186742272.0


## 14. Build real input with the training preprocessing contract

The simulated training pipeline generated noise and PSD from the same `NoiseModel`. For real data, the equivalent step is to estimate a detector-specific PSD from off-source real strain and pass it to the same `SignalProcessor`.

In [37]:
# ------------------------------------------------------------
# 14.0 Build SignalProcessor with the same config as M08 training
# ------------------------------------------------------------

from src.config import SimulationConfig
from src.processing import SignalProcessor

# These values should already be defined from the generation config.
# We keep them explicit here to avoid hidden notebook-state errors.
fs = 4096
duration = 4.0
context_start = 1664
context_end = 1664

sim_config = SimulationConfig(
    duration=duration,
    processing_context_start_samples=context_start,
    processing_context_end_samples=context_end,
)

processor = SignalProcessor(
    config=sim_config,
    **gen_cfg["signal_processor"],
)

print("SignalProcessor ready")
print("whitening_method:", processor.whitening_method)
print("apply_highpass:", processor.apply_highpass)
print("apply_lowpass:", processor.apply_lowpass)
print("apply_standardization:", processor.apply_standardization)
print("output_mode:", processor.output_mode)
print("highpass_frequency:", processor.highpass_frequency)
print("lowpass_frequency:", processor.lowpass_frequency)
print("processing_length:", sim_config.processing_length)
print("output_length:", sim_config.length)

assert processor.whitening_method == "psd"
assert processor.apply_highpass is True
assert processor.apply_lowpass is True
assert processor.apply_standardization is False
assert processor.output_mode == "crop_to_config"
assert sim_config.processing_length == 19712
assert sim_config.length == 16384

SignalProcessor ready
whitening_method: psd
apply_highpass: True
apply_lowpass: True
apply_standardization: False
output_mode: crop_to_config
highpass_frequency: 30.0
lowpass_frequency: 512.0
processing_length: 19712
output_length: 16384


In [39]:
from pycbc.psd import interpolate, inverse_spectrum_truncation

final_duration = float(duration)

final_length = int(final_duration * fs)
processing_length = final_length + context_start + context_end

processing_duration = processing_length / fs
processing_delta_f = 1.0 / processing_duration
processing_flength = processing_length // 2 + 1

print("final_length:", final_length)
print("processing_length:", processing_length)
print("processing_duration:", processing_duration)
print("processing_delta_f:", processing_delta_f)
print("processing_flength:", processing_flength)

assert final_length == 16384
assert processing_length == 19712

final_length: 16384
processing_length: 19712
processing_duration: 4.8125
processing_delta_f: 0.2077922077922078
processing_flength: 9857


In [40]:
def estimate_offsource_psd_long(
    strain,
    event_time,
    psd_start_offset=-512.0,
    psd_end_offset=-128.0,
    psd_segment_duration=8.0,
    low_frequency_cutoff=30.0,
    max_filter_duration=0.5,
):
    psd_start = event_time + psd_start_offset
    psd_end = event_time + psd_end_offset

    available_start = float(strain.start_time)
    available_end = float(strain.end_time)

    if psd_start < available_start or psd_end > available_end:
        raise ValueError(
            f"PSD window [{psd_start}, {psd_end}] outside available "
            f"[{available_start}, {available_end}]"
        )

    psd_data = strain.time_slice(psd_start, psd_end)

    psd = psd_data.psd(psd_segment_duration)
    psd = interpolate(psd, processing_delta_f)

    max_filter_len = int(round(max_filter_duration * fs))

    psd = inverse_spectrum_truncation(
        psd,
        max_filter_len=max_filter_len,
        low_frequency_cutoff=low_frequency_cutoff,
        trunc_method="hann",
    )

    if len(psd) > processing_flength:
        psd = psd[:processing_flength]
    elif len(psd) < processing_flength:
        raise ValueError(f"PSD too short: {len(psd)} < {processing_flength}")

    if not np.all(np.isfinite(psd.numpy())):
        raise ValueError("PSD contains non-finite values.")

    return psd

In [41]:
def build_real_input_like_training(
    *,
    raw_strains_long,
    center_time,
    processor,
    psd_start_offset=-512.0,
    psd_end_offset=-128.0,
    psd_segment_duration=8.0,
):
    """
    Build one real HLV input using the same preprocessing contract as M08 training.

    The final 4 s output window is centered at `center_time`.
    The input to SignalProcessor includes the same processing context used in training.
    """
    output_start = center_time - final_duration / 2
    output_end = center_time + final_duration / 2

    processing_start = output_start - context_start / fs
    processing_end = output_end + context_end / fs

    real_segments = {}

    for ifo in REAL_DETECTORS:
        seg = raw_strains_long[ifo].time_slice(processing_start, processing_end)

        if len(seg) != processing_length:
            raise ValueError(
                f"{ifo}: expected {processing_length}, got {len(seg)}"
            )

        real_segments[ifo] = seg

    psds = {}

    for ifo in REAL_DETECTORS:
        psds[ifo] = estimate_offsource_psd_long(
            raw_strains_long[ifo],
            event_time=center_time,
            psd_start_offset=psd_start_offset,
            psd_end_offset=psd_end_offset,
            psd_segment_duration=psd_segment_duration,
            low_frequency_cutoff=30.0,
            max_filter_duration=0.5,
        )

    processed = processor.process_network(
        strains=real_segments,
        psds=psds,
    )

    X = np.stack(
        [processed[ifo].numpy() for ifo in REAL_DETECTORS],
        axis=0,
    ).astype(np.float32)

    X = X[None, :, :]

    if X.shape != (1, 3, 16384):
        raise ValueError(f"Bad X shape: {X.shape}")

    if not np.all(np.isfinite(X)):
        raise ValueError("X contains non-finite values.")

    return X, processed, psds, real_segments

In [42]:
X_real, processed_real, psds_real, real_segments = build_real_input_like_training(
    raw_strains_long=raw_strains_4096,
    center_time=event_time,
    processor=processor,
)

/afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/lib/python3.10/site-packages/pycbc/waveform/plugin.py:99: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
print("X_real shape:", X_real.shape)
print("finite:", np.all(np.isfinite(X_real)))
print("mean:", float(X_real.mean()))
print("std:", float(X_real.std()))

assert X_real.shape == (1, 3, 16384)
assert np.all(np.isfinite(X_real))

## 15. Real/synthetic scale diagnostic

In [43]:
import h5py

dataset_path = "/data/vserrano/cbc_pe_data/processed/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000/bbh_processed_4s_seobnrv4opt_snr10-25_n500_000.h5"
rng = np.random.default_rng(123)
n_probe = 5000

with h5py.File(dataset_path, "r") as f:
    n_total = f["X"].shape[0]
    probe_idx = np.sort(rng.choice(n_total, size=n_probe, replace=False))
    X_probe = f["X"][probe_idx].astype(np.float32)

rows = []

for j, ifo in enumerate(REAL_DETECTORS):
    channel_std = X_probe[:, j, :].std(axis=1)
    channel_mean = X_probe[:, j, :].mean(axis=1)
    channel_maxabs = np.max(np.abs(X_probe[:, j, :]), axis=1)

    rows.append({
        "detector": ifo,
        "train_mean_of_means": float(np.mean(channel_mean)),
        "train_median_std": float(np.median(channel_std)),
        "train_q05_std": float(np.quantile(channel_std, 0.05)),
        "train_q95_std": float(np.quantile(channel_std, 0.95)),
        "train_median_maxabs": float(np.median(channel_maxabs)),
        "train_q95_maxabs": float(np.quantile(channel_maxabs, 0.95)),
    })

train_scale_df = pd.DataFrame(rows)
display(train_scale_df)

,detector,train_mean_of_means,train_median_std,train_q05_std,train_q95_std,train_median_maxabs,train_q95_maxabs
0,H1,-0.000224,22.066248,21.580492,22.624434,92.406082,122.905823
1,L1,-0.000067,22.060675,21.584286,22.673035,92.248581,124.518295
2,V1,0.000006,22.005814,21.533955,22.671404,91.671600,129.735229


In [44]:
def summarize_real_scale(X, tag):
    rows = []

    for j, ifo in enumerate(REAL_DETECTORS):
        arr = X[0, j]

        rows.append({
            "preprocessing": tag,
            "detector": ifo,
            "real_mean": float(np.mean(arr)),
            "real_std": float(np.std(arr)),
            "real_maxabs": float(np.max(np.abs(arr))),
            "finite": bool(np.all(np.isfinite(arr))),
        })

    return pd.DataFrame(rows)

In [45]:
real_scale_df = summarize_real_scale(
    X_real,
    tag="offsource_psd_4096s_centered",
)

scale_comparison_df = real_scale_df.merge(
    train_scale_df,
    on="detector",
    how="left",
)

scale_comparison_df["std_ratio_real_to_train_median"] = (
    scale_comparison_df["real_std"] / scale_comparison_df["train_median_std"]
)

scale_comparison_df["maxabs_ratio_real_to_train_q95"] = (
    scale_comparison_df["real_maxabs"] / scale_comparison_df["train_q95_maxabs"]
)

display(scale_comparison_df)

,preprocessing,detector,real_mean,real_std,real_maxabs,finite,train_mean_of_means,train_median_std,train_q05_std,train_q95_std,train_median_maxabs,train_q95_maxabs,std_ratio_real_to_train_median,maxabs_ratio_real_to_train_q95
0,offsource_psd_4096s_centered,H1,0.001536,23.639956,105.104576,True,-0.000224,22.066248,21.580492,22.624434,92.406082,122.905823,1.071317,0.855164
1,offsource_psd_4096s_centered,L1,-0.012151,86.263588,206.907181,True,-0.000067,22.060675,21.584286,22.673035,92.248581,124.518295,3.910288,1.661661
2,offsource_psd_4096s_centered,V1,0.023862,154.776749,473.826996,True,0.000006,22.005814,21.533955,22.671404,91.671600,129.735229,7.033448,3.652262


## 16. M08 + Mondrian inference

In [46]:
def run_m08_mondrian_on_X(X, tag):
    pred_std, pred_phys, emb = predict_m08(model, X)

    lower_std, upper_std, bins, scores = apply_selected_mondrian_systems(
        pred_std=pred_std,
        emb=emb,
        selected_mondrian_systems=selected_mondrian_systems,
    )

    lower_phys, upper_phys = intervals_std_to_phys(
        lower_std,
        upper_std,
    )

    rows = []

    for j, name in enumerate(label_names):
        rows.append({
            "preprocessing": tag,
            "event": event_name,
            "label": name,
            "pred": float(pred_phys[0, j]),
            "lower": float(lower_phys[0, j]),
            "upper": float(upper_phys[0, j]),
            "width": float(upper_phys[0, j] - lower_phys[0, j]),
            "bin": int(bins[0, j]),
            "taxonomy": selected_mondrian_systems[name].taxonomy_mode,
            "interval_mode": selected_mondrian_systems[name].interval_mode,
        })

    return pd.DataFrame(rows), pred_std, pred_phys, emb, bins, scores

In [47]:
real_result_df, pred_real_std, pred_real_phys, emb_real, bins_real, scores_real = (
    run_m08_mondrian_on_X(
        X_real,
        tag="offsource_psd_4096s_centered",
    )
)

display(real_result_df)

,preprocessing,event,label,pred,lower,upper,width,bin,taxonomy,interval_mode
0,offsource_psd_4096s_centered,GW170814,chirp_mass,3.364545,1.79425,5.980965,4.186716,0,prediction,asymmetric
1,offsource_psd_4096s_centered,GW170814,total_mass,9.598968,1.37262,23.059301,21.686680,0,prediction,asymmetric
2,offsource_psd_4096s_centered,GW170814,chi_eff,0.469226,0.07282,0.865631,0.792811,5,difficulty,symmetric


The intervals are calibrated on synthetic data. Therefore, for real data they should be interpreted as a transfer diagnostic unless the real input distribution is shown to match the synthetic calibration distribution sufficiently well.

## 17. PSD-window sensitivity

In [49]:
PSD_WINDOWS = [
    (-1600.0, -1216.0),
    (-1400.0, -1016.0),
    (-1200.0, -816.0),
    (-1024.0, -640.0),
    (-768.0, -384.0),
    (-512.0, -128.0),
]

psd_sensitivity_results = []
psd_sensitivity_scales = []

for psd_start_offset, psd_end_offset in PSD_WINDOWS:
    tag = f"psd_{int(psd_start_offset)}_{int(psd_end_offset)}"

    print("\n", "=" * 80)
    print(tag)

    X_tmp, processed_tmp, psds_tmp, segs_tmp = build_real_input_like_training(
        raw_strains_long=raw_strains_4096,
        center_time=event_time,
        processor=processor,
        psd_start_offset=psd_start_offset,
        psd_end_offset=psd_end_offset,
        psd_segment_duration=8.0,
    )

    scale_tmp = summarize_real_scale(X_tmp, tag=tag)
    result_tmp, *_ = run_m08_mondrian_on_X(X_tmp, tag=tag)

    psd_sensitivity_scales.append(scale_tmp)
    psd_sensitivity_results.append(result_tmp)

psd_sensitivity_scale_df = pd.concat(psd_sensitivity_scales, ignore_index=True)
psd_sensitivity_result_df = pd.concat(psd_sensitivity_results, ignore_index=True)

display(psd_sensitivity_scale_df)
display(psd_sensitivity_result_df)


psd_-1600_-1216

psd_-1400_-1016

psd_-1200_-816

psd_-1024_-640

psd_-768_-384

psd_-512_-128


,preprocessing,detector,real_mean,real_std,real_maxabs,finite
0,psd_-1600_-1216,H1,0.001445,23.399437,102.653603,True
1,psd_-1600_-1216,L1,-0.011879,85.420967,205.594559,True
2,psd_-1600_-1216,V1,0.023588,153.389740,468.150482,True
3,psd_-1400_-1016,H1,0.001487,23.486172,103.161880,True
4,psd_-1400_-1016,L1,-0.011685,84.679619,204.158203,True
5,psd_-1400_-1016,V1,0.024349,159.461594,483.436249,True
6,psd_-1200_-816,H1,0.001483,23.519014,104.336288,True
7,psd_-1200_-816,L1,-0.011644,84.854294,204.705948,True
8,psd_-1200_-816,V1,0.024335,160.171844,484.339600,True
9,psd_-1024_-640,H1,0.001268,23.615892,104.809555,True


,preprocessing,event,label,pred,lower,upper,width,bin,taxonomy,interval_mode
0,psd_-1600_-1216,GW170814,chirp_mass,3.306173,1.735877,5.922593,4.186716,0,prediction,asymmetric
1,psd_-1600_-1216,GW170814,total_mass,9.455883,1.229536,22.916216,21.686680,0,prediction,asymmetric
2,psd_-1600_-1216,GW170814,chi_eff,0.498776,0.102370,0.895181,0.792811,5,difficulty,symmetric
3,psd_-1400_-1016,GW170814,chirp_mass,3.374434,1.804138,5.990854,4.186716,0,prediction,asymmetric
4,psd_-1400_-1016,GW170814,total_mass,9.632550,1.406202,23.092883,21.686680,0,prediction,asymmetric
5,psd_-1400_-1016,GW170814,chi_eff,0.478711,0.082306,0.875117,0.792811,5,difficulty,symmetric
6,psd_-1200_-816,GW170814,chirp_mass,3.380235,1.809939,5.996655,4.186716,0,prediction,asymmetric
7,psd_-1200_-816,GW170814,total_mass,9.723903,1.497555,23.184236,21.686680,0,prediction,asymmetric
8,psd_-1200_-816,GW170814,chi_eff,0.447263,0.050857,0.843668,0.792811,5,difficulty,symmetric
9,psd_-1024_-640,GW170814,chirp_mass,3.306476,1.736180,5.922896,4.186716,0,prediction,asymmetric


In [51]:
display(
    psd_sensitivity_result_df.pivot_table(
        index="preprocessing",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

display(
    psd_sensitivity_scale_df.pivot_table(
        index="preprocessing",
        columns="detector",
        values="real_std",
        aggfunc="first",
    )
)

label,chi_eff,chirp_mass,total_mass
preprocessing,,,
psd_-1024_-640,0.487087,3.306476,9.465739
psd_-1200_-816,0.447263,3.380235,9.723903
psd_-1400_-1016,0.478711,3.374434,9.632550
psd_-1600_-1216,0.498776,3.306173,9.455883
psd_-512_-128,0.469226,3.364545,9.598968
psd_-768_-384,0.496115,3.251719,9.311155


detector,H1,L1,V1
preprocessing,,,
psd_-1024_-640,23.615892,86.220123,156.278931
psd_-1200_-816,23.519014,84.854294,160.171844
psd_-1400_-1016,23.486172,84.679619,159.461594
psd_-1600_-1216,23.399437,85.420967,153.389740
psd_-512_-128,23.639956,86.263588,154.776749
psd_-768_-384,23.522486,85.316635,154.075317


The prediction is stable under reasonable off-source PSD-window choices, but remains physically inconsistent with GW170814. This suggests that the failure is not mainly caused by the PSD-window selection, but by a deeper real/synthetic domain mismatch, timing mismatch, or model-transfer issue.

## 18. Center-time sensitivity

In [52]:
CENTER_OFFSETS = np.array([
    -1.50, -1.25, -1.00, -0.75, -0.50, -0.25,
     0.00,
     0.25,  0.50,  0.75,  1.00,  1.25,  1.50,
])

time_sensitivity_results = []
time_sensitivity_scales = []

for dt_center in CENTER_OFFSETS:
    center_time = event_time + float(dt_center)
    tag = f"center_{dt_center:+.2f}s"

    print("\n", "=" * 80)
    print(tag)

    X_tmp, processed_tmp, psds_tmp, segs_tmp = build_real_input_like_training(
        raw_strains_long=raw_strains_4096,
        center_time=center_time,
        processor=processor,
        psd_start_offset=-512.0,
        psd_end_offset=-128.0,
        psd_segment_duration=8.0,
    )

    scale_tmp = summarize_real_scale(X_tmp, tag=tag)
    result_tmp, *_ = run_m08_mondrian_on_X(X_tmp, tag=tag)

    scale_tmp["center_offset_s"] = dt_center
    result_tmp["center_offset_s"] = dt_center

    time_sensitivity_scales.append(scale_tmp)
    time_sensitivity_results.append(result_tmp)

time_sensitivity_scale_df = pd.concat(time_sensitivity_scales, ignore_index=True)
time_sensitivity_result_df = pd.concat(time_sensitivity_results, ignore_index=True)

display(time_sensitivity_scale_df)
display(time_sensitivity_result_df)


center_-1.50s

center_-1.25s

center_-1.00s

center_-0.75s

center_-0.50s

center_-0.25s

center_+0.00s

center_+0.25s

center_+0.50s

center_+0.75s

center_+1.00s

center_+1.25s

center_+1.50s


,preprocessing,detector,real_mean,real_std,real_maxabs,finite,center_offset_s
0,center_-1.50s,H1,-0.002133,23.750530,104.469772,True,-1.50
1,center_-1.50s,L1,0.009805,85.276680,201.037064,True,-1.50
2,center_-1.50s,V1,0.011863,155.085251,431.133331,True,-1.50
3,center_-1.25s,H1,0.009857,23.642326,104.425415,True,-1.25
4,center_-1.25s,L1,0.023179,85.589439,201.558517,True,-1.25
5,center_-1.25s,V1,-0.002558,153.406326,430.971039,True,-1.25
6,center_-1.00s,H1,0.007834,23.566851,104.531822,True,-1.00
7,center_-1.00s,L1,0.009051,85.777832,202.065399,True,-1.00
8,center_-1.00s,V1,-0.026842,152.741684,430.526337,True,-1.00
9,center_-0.75s,H1,0.018571,23.599737,104.444466,True,-0.75


,preprocessing,event,label,pred,lower,upper,width,bin,taxonomy,interval_mode,center_offset_s
0,center_-1.50s,GW170814,chirp_mass,2.959614,1.389319,5.576034,4.186716,0,prediction,asymmetric,-1.50
1,center_-1.50s,GW170814,total_mass,8.243298,0.016950,21.703631,21.686680,0,prediction,asymmetric,-1.50
2,center_-1.50s,GW170814,chi_eff,0.378116,-0.018289,0.774522,0.792811,5,difficulty,symmetric,-1.50
3,center_-1.25s,GW170814,chirp_mass,2.941834,1.371538,5.558254,4.186716,0,prediction,asymmetric,-1.25
4,center_-1.25s,GW170814,total_mass,7.974122,-0.252226,21.434455,21.686680,0,prediction,asymmetric,-1.25
5,center_-1.25s,GW170814,chi_eff,0.477039,0.080633,0.873444,0.792811,5,difficulty,symmetric,-1.25
6,center_-1.00s,GW170814,chirp_mass,2.919482,1.349187,5.535902,4.186716,0,prediction,asymmetric,-1.00
7,center_-1.00s,GW170814,total_mass,7.694950,-0.531398,21.155283,21.686680,0,prediction,asymmetric,-1.00
8,center_-1.00s,GW170814,chi_eff,0.464514,0.068108,0.860919,0.792811,5,difficulty,symmetric,-1.00
9,center_-0.75s,GW170814,chirp_mass,3.298879,1.728583,5.915299,4.186716,0,prediction,asymmetric,-0.75


In [53]:
display(
    time_sensitivity_result_df.pivot_table(
        index="center_offset_s",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

display(
    time_sensitivity_scale_df.pivot_table(
        index="center_offset_s",
        columns="detector",
        values="real_std",
        aggfunc="first",
    )
)

label,chi_eff,chirp_mass,total_mass
center_offset_s,,,
-1.50,0.378116,2.959614,8.243298
-1.25,0.477039,2.941834,7.974122
-1.00,0.464514,2.919482,7.694950
-0.75,0.475244,3.298879,8.857604
-0.50,0.445733,3.054302,8.040625
-0.25,0.434500,3.204674,9.090290
0.00,0.469226,3.364545,9.598968
0.25,0.457330,3.231526,8.946611
0.50,0.484505,3.234438,8.996963


detector,H1,L1,V1
center_offset_s,,,
-1.50,23.750530,85.276680,155.085251
-1.25,23.642326,85.589439,153.406326
-1.00,23.566851,85.777832,152.741684
-0.75,23.599737,87.280190,151.946182
-0.50,23.615847,86.769058,152.569275
-0.25,23.582626,86.739029,153.733734
0.00,23.639956,86.263588,154.776749
0.25,23.574713,86.460243,153.467804
0.50,23.482624,86.824142,152.618958


## 19. Off-source real-noise controls

In [54]:
NOISE_CENTER_OFFSETS = np.array([
    -1800.0, -1600.0, -1400.0, -1200.0,
    -1000.0, -800.0, -600.0, -400.0,
    200.0, 300.0,
])

noise_control_results = []
noise_control_scales = []

for dt_center in NOISE_CENTER_OFFSETS:
    center_time = event_time + float(dt_center)
    tag = f"noise_center_{dt_center:+.0f}s"

    print("\n", "=" * 80)
    print(tag)

    X_tmp, processed_tmp, psds_tmp, segs_tmp = build_real_input_like_training(
        raw_strains_long=raw_strains_4096,
        center_time=center_time,
        processor=processor,
        psd_start_offset=-512.0,
        psd_end_offset=-128.0,
        psd_segment_duration=8.0,
    )

    scale_tmp = summarize_real_scale(X_tmp, tag=tag)
    result_tmp, *_ = run_m08_mondrian_on_X(X_tmp, tag=tag)

    scale_tmp["center_offset_s"] = dt_center
    result_tmp["center_offset_s"] = dt_center
    result_tmp["is_event_window"] = False

    noise_control_scales.append(scale_tmp)
    noise_control_results.append(result_tmp)

noise_control_scale_df = pd.concat(noise_control_scales, ignore_index=True)
noise_control_result_df = pd.concat(noise_control_results, ignore_index=True)

display(
    noise_control_result_df.pivot_table(
        index="center_offset_s",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

display(
    noise_control_scale_df.pivot_table(
        index="center_offset_s",
        columns="detector",
        values="real_std",
        aggfunc="first",
    )
)


noise_center_-1800s

noise_center_-1600s

noise_center_-1400s

noise_center_-1200s

noise_center_-1000s

noise_center_-800s

noise_center_-600s

noise_center_-400s

noise_center_+200s

noise_center_+300s


label,chi_eff,chirp_mass,total_mass
center_offset_s,,,
-1800.0,-0.395114,1.150117,5.257635
-1600.0,-0.392227,1.346846,5.281461
-1400.0,-0.495068,0.760321,4.685726
-1200.0,-0.343086,1.107890,5.198072
-1000.0,-0.380316,1.517467,6.224630
-800.0,-0.364946,2.087398,7.744798
-600.0,-0.423450,1.728416,6.366385
-400.0,-0.285239,1.903069,6.062743
200.0,-0.227007,2.422489,7.600160


detector,H1,L1,V1
center_offset_s,,,
-1800.0,22.193399,147.691299,154.768341
-1600.0,22.275991,142.675903,149.027039
-1400.0,23.162254,134.405777,147.932312
-1200.0,22.300331,126.529396,143.947693
-1000.0,23.320801,116.893829,145.845825
-800.0,23.333941,108.154617,146.699921
-600.0,22.593122,102.958382,139.240295
-400.0,22.803259,97.744965,149.166687
200.0,23.853666,79.646469,141.885147


## 20. Detector ablation

In [55]:
def run_detector_ablation(X_base):
    variants = {}

    X = X_base.copy()
    variants["HLV"] = X

    X_h1 = X_base.copy()
    X_h1[:, 1, :] = 0.0
    X_h1[:, 2, :] = 0.0
    variants["H1_only"] = X_h1

    X_hl = X_base.copy()
    X_hl[:, 2, :] = 0.0
    variants["H1_L1"] = X_hl

    X_hv = X_base.copy()
    X_hv[:, 1, :] = 0.0
    variants["H1_V1"] = X_hv

    rows = []

    for tag, X_var in variants.items():
        result_df, *_ = run_m08_mondrian_on_X(X_var, tag=tag)
        rows.append(result_df)

    return pd.concat(rows, ignore_index=True)

ablation_result_df = run_detector_ablation(X_real)

display(
    ablation_result_df.pivot_table(
        index="preprocessing",
        columns="label",
        values="pred",
        aggfunc="first",
    )
)

label,chi_eff,chirp_mass,total_mass
preprocessing,,,
H1_L1,0.102703,26.047491,60.894250
H1_V1,0.580722,3.579912,10.806531
H1_only,-0.141155,23.454122,52.611585
HLV,0.469226,3.364545,9.598968


## Interim conclusion

The clean real-event pipeline now applies the same preprocessing contract used in M08 training: H1/L1/V1 ordering, 4096 Hz sampling, 19712-sample processing context, PSD whitening, 30 Hz high-pass, 512 Hz low-pass, no per-channel standardization, and final 16384-sample crop.

The synthetic M08 + Mondrian reconstruction behaves correctly on the held-out synthetic test set.

For GW170814, long off-source PSD whitening substantially reduces the real/synthetic scale mismatch compared with analytical PSD whitening. However, L1 and V1 remain several times above the synthetic training scale. The M08 prediction is stable under reasonable PSD-window changes and center-time shifts, but remains physically inconsistent with GW170814, with masses pushed to the lowest prediction bin.

This suggests that the current failure is not mainly due to PSD-window choice or event-window centering, but rather to real/synthetic domain mismatch, detector-channel scale mismatch, or lack of robustness of M08 to real detector noise.

In [56]:
X_real_channel_z = X_real.copy()

for j in range(3):
    mu = X_real_channel_z[:, j, :].mean(axis=1, keepdims=True)
    sig = X_real_channel_z[:, j, :].std(axis=1, keepdims=True)
    X_real_channel_z[:, j, :] = (X_real_channel_z[:, j, :] - mu) / (sig + 1e-8)

z_result_df, *_ = run_m08_mondrian_on_X(
    X_real_channel_z,
    tag="diagnostic_per_channel_zscore_not_training_compatible",
)

display(z_result_df)

,preprocessing,event,label,pred,lower,upper,width,bin,taxonomy,interval_mode
0,diagnostic_per_channel_zscore_not_training_com...,GW170814,chirp_mass,26.627572,21.951843,32.707767,10.755925,7,prediction,asymmetric
1,diagnostic_per_channel_zscore_not_training_com...,GW170814,total_mass,61.260576,49.739670,72.980387,23.240717,2,prediction,asymmetric
2,diagnostic_per_channel_zscore_not_training_com...,GW170814,chi_eff,0.096916,-0.215950,0.409782,0.625732,4,difficulty,symmetric


In [57]:
display(real_result_df)

,preprocessing,event,label,pred,lower,upper,width,bin,taxonomy,interval_mode
0,offsource_psd_4096s_centered,GW170814,chirp_mass,3.364545,1.79425,5.980965,4.186716,0,prediction,asymmetric
1,offsource_psd_4096s_centered,GW170814,total_mass,9.598968,1.37262,23.059301,21.686680,0,prediction,asymmetric
2,offsource_psd_4096s_centered,GW170814,chi_eff,0.469226,0.07282,0.865631,0.792811,5,difficulty,symmetric


The failure of the baseline M08 model on GW170814 is primarily driven by detector-dependent real/synthetic scale mismatch, especially in V1. This is supported by three observations: (i) the HLV prediction collapses to the lowest mass bin when V1 is included, (ii) H1-only and H1+L1 ablations yield physically plausible mass estimates, and (iii) a diagnostic per-channel z-score transformation moves the HLV prediction to a plausible mass range.

However, the z-score result cannot be interpreted as a calibrated physical estimate because M08 was not trained or conformally calibrated under this input normalization. The scientifically rigorous next step is to train and recalibrate a model using the same input normalization or scale-robust augmentation throughout train, validation, calibration, test, and real-event inference.